# JRC hazards — catalog explorer (no network)

The `earthlens.jrc` backend serves every JRC / Copernicus-EMS hazard product from
one class, selected by dataset and dispatched on the catalog row's `kind`:

- **`flood_hazard_raster`** — the European Flood Hazard Map (EFHM): river-flood
  **water depth (m)** for a set of **return periods** over Europe and the
  Mediterranean.
- **`sea_level_gridded`** — the probabilistic **Total Water Level** forecasts
  (medium-term and subseasonal), global 0.25° NetCDF cubes.
- **`sea_level_coastal`** — the subseasonal global per-country coastal summary.

This notebook inspects the catalog and the URL / pixel-window helpers without
touching the network.

## Installation

`earthlens` is a namespace package split across one core distribution and five
thematic provider distributions, so you install only the providers you need.

**Just the JRC products** — the EFHM flood-hazard maps and the sea-level (TWL)
forecasts both live in the `hazards` provider:

```bash
pip install earthlens-hazards
```

That is the whole install for this notebook series. It pulls `earthlens-core`
(and through it pyramids, the GIS backend) automatically, and it needs **no
optional extra**: every JRC product is plain anonymous HTTPS from the JRC's
public server, read with the core HTTP client and pyramids. The `hazards`
distribution does publish extras — `emdat`, `fdsn`, `hdx`, `osm`, `osm-pbf`,
`overture` — but those belong to *other* backends in the same theme and are not
needed here.

**Everything** — all five providers, if you want the other 40-odd backends:

```bash
pip install earthlens
```

Add the curated SDK bundle with `pip install "earthlens[all]"` when you also
want the backends that need a third-party SDK (Earth Engine, CDS, Copernicus
Marine, and so on). Again, none of that is required for JRC.

Python 3.11 or newer is required.

```python
# Verify the install: this is the entire dependency surface this series uses.
from earthlens.core import EarthLens
from earthlens.jrc import Catalog
```

## The command line

Installing `earthlens-hazards` also puts an **`earthlens`** console script on your
PATH. It is a *catalog* tool: it answers "what exists, what does it hold, is it
still valid" without downloading anything and without network access. Fetching
data is the Python API shown in the next notebooks — there is no `earthlens
download` command.

Two command groups, `datasets` and `providers`:

```bash
earthlens --help
earthlens datasets --help
```

### Which keys does the JRC backend answer to?

```bash
earthlens providers list
```

The `jrc` row lists every facade key at once — the bare `jrc`, plus `efhm`,
`jrc-flood`,
`jrc-flood-hazard`, `jrc-sea-level`, `jrc:coastal-forecast`,
`jrc:european-flood-hazard`, `jrc:sea-level-forecast`, `jrc:twl-forecast` — which
is the quickest way to check what a given `data_source=` string resolves to.

In [ ]:
import sys

# On a shell you type `earthlens ...`; here the module form is used so the
# command is guaranteed to run inside *this* kernel's environment rather than
# whichever `earthlens` happens to be first on PATH. The two are equivalent.
CLI = f"{sys.executable} -m earthlens.cli"

# Every command in this section is offline: it reads the catalog that shipped
# inside the package.
!{CLI} datasets list --provider jrc

### Inspecting one dataset

`where` answers "who serves this?" across every installed provider — useful when a
name such as `elevation` is served by more than one backend. `show` prints the
whole catalog record: band, units, CRS, nodata, the URL template, and for the
sea-level rows the cadence, horizon and default field.

```bash
earthlens datasets where efhm
earthlens datasets show jrc efhm
earthlens datasets show jrc sea_level_medium_term
```

In [ ]:
!{CLI} datasets where efhm

### Searching and filtering

`search` is free text plus facets. `facets` tells you what you are allowed to
filter on and how many distinct values each has, so you are not guessing at
filter names.

```bash
earthlens datasets facets --provider jrc
earthlens datasets search flood --provider jrc
earthlens datasets search --provider jrc --filter cadence=weekly
```

In [ ]:
!{CLI} datasets facets --provider jrc
!{CLI} datasets search --provider jrc --filter cadence=weekly

### Machine-readable output

Three flags make the same query scriptable — `--json` for a JSON array, `--ids-only`
for bare `provider<TAB>id` lines to pipe, and `--count` for just a number.

```bash
earthlens datasets search --provider jrc --json
earthlens datasets search --provider jrc --ids-only
earthlens datasets search --provider jrc --count
```

In [ ]:
!{CLI} datasets search --provider jrc --ids-only
!{CLI} datasets search --provider jrc --count --json

### Checking the catalog is sound

`validate` checks each curated row against the rules its `kind` requires — a
gridded sea-level row must carry a `base_url`, `product` and glob, and so on. It
is offline, so it is the one to reach for first.

`audit` is the *live* counterpart: it re-reads the provider's upstream index and
reports drift. **JRC has no public listing endpoint**, so it reports `unsupported`
rather than pretending to check — that is expected, not a failure.

```bash
earthlens datasets validate jrc
earthlens datasets audit jrc          # -> unsupported, by design
```

In [ ]:
!{CLI} datasets validate jrc

## The same questions from Python

Everything the CLI answers is available on the facade, which is what you want
inside a script. These are classmethods — they take the `data_source` key, so you
do not have to build a request first.

`options_for` is the one worth remembering: it reports the keyword arguments a
backend actually accepts, so you never have to guess at a parameter name or read
the source.

In [ ]:
from earthlens.core import EarthLens, find, sources

# Which keyword arguments does this backend take, beyond the facade's own?
print("options :", sorted(EarthLens.options_for("jrc:sea-level-forecast")))

# Which datasets does it serve, and what is in one of them?
print("datasets:", EarthLens.list_datasets("jrc:sea-level-forecast"))

# Free-text search *within* one backend, when you half-remember a name.
print("'coastal' ->", EarthLens.guess_dataset("efhm", "coastal"))

record = EarthLens.describe_dataset("efhm", "sea_level_medium_term")
print("cadence :", record.cadence, "| horizon:", record.horizon_days, "days")

### Finding JRC among the other providers

`find()` searches every installed provider at once, which is how you discover that
a subject you care about is served from more than one place — flood data, for
instance, comes from several backends with different coverage and resolution.

In [ ]:
matches = find("flood")
print()
print(len(sources()), "data sources are installed in total.")

## The catalog

Four datasets across the three kinds. The EFHM is addressed by return period; the
sea-level rows are addressed by forecast cycle.

In [ ]:
from earthlens.jrc import Catalog

catalog = Catalog()
for name in sorted(catalog.datasets):
    row = catalog.get(name)
    print(f"{name:32s} kind={row.kind:22s} units={row.units or '-':4s} crs={row.crs}")

print("------------")
efhm = catalog.get("efhm")
print("efhm return periods:", efhm.return_periods)
sea = catalog.get("sea_level_medium_term")
print(
    "medium-term:",
    sea.cadence,
    "| horizon:",
    sea.horizon_days,
    "days",
    "| default field:",
    sea.default_field,
)
print("licence:", catalog.license_id)

## How a small AOI stays cheap

Each return period is one whole-Europe GeoTIFF (~23 GB uncompressed). The backend
never reads it whole: it opens the file lazily and reads only the AOI's pixel
window over `/vsicurl` (HTTP range requests) via pyramids' `Dataset.crop(bbox=)`.
Building the request — the per-return-period URL below — is offline; only
`.download()` touches the network.

In [ ]:
from earthlens.jrc._helpers import efhm_url

# One whole-Europe ~23 GB GeoTIFF per return period; a small AOI reads only its
# pixel window via pyramids' Dataset.crop(bbox=) over /vsicurl (a few hundred KB).
for rp in (100, 200, 500):
    print(f"RP{rp}:", efhm_url(rp))

## Licence and attribution

Every JRC product here is **CC-BY-4.0** — permissive, but attribution is a
condition of use, not a courtesy. The catalog carries both the licence id and the
citation string, so a script that publishes a figure can reproduce the credit
without anyone re-typing it.

In [ ]:
print("licence:", catalog.license_id)
print()
print("cite as:")
print(" ", catalog.attribution)

## Next

See **EFHM quickstart** for a real windowed download and map, and
**Sea-level TWL forecast** for the coastal forecast products.